# Multi-Modal Binary Ensemble (T1 + FLAIR) su OpenNeuro FCD
Questo notebook esegue il test combinando le predizioni di due reti SFCN addestrate separatamente (una su T1 e una su FLAIR). 
Corregge inoltre la visualizzazione clinica, forzando la classe `fcd` a essere la classe Positiva (1) per l'AUC e la Confusion Matrix.

In [ ]:
!rm -rf SFCN
!git clone https://github.com/PietroSchgor/SFCN.git

import sys
sys.path.append('./SFCN')

In [ ]:
import os
import json
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

from dp_model.model_files.sfcn import SFCN

PLOTS_DIR = '/kaggle/working/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

## 1. Configurazione

In [ ]:
KAGGLE_DATA_DIR = "/kaggle/input/corrected-fcd-subject/corrected_FCD_subject" # Aggiorna col path root

MODEL_T1_PATH = "/kaggle/input/sfcn-models/best_model_t1.pth" # Aggiorna col path del modello T1
MODEL_FLAIR_PATH = "/kaggle/input/sfcn-models/best_model_flair.pth" # Aggiorna col path del modello FLAIR

OUTPUT_DIM = 2
CUSTOM_CHANNELS = [28, 58, 128, 256, 256, 64] # Il trick applicato nel training


## 2. Multi-Modal Dataloader (T1 e FLAIR allineati)

In [ ]:
class OpenNeuroMultiModalDataset(Dataset):
    def __init__(self, data_dir):
        self.samples = []
        
        if not os.path.exists(data_dir):
            print(f"ATTENZIONE: Directory {data_dir} non trovata.")
            return
            
        subjects = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d)) and d.startswith("sub-")]
        
        raw_samples = []
        unique_groups = set()
        
        for sub in subjects:
            # CERCA IMMAGINE T1
            nii_path_t1 = os.path.join(data_dir, sub, f"{sub}_T1w_MNI152_1mm.nii")
            if not os.path.exists(nii_path_t1):
                nii_path_t1 += ".gz"
                if not os.path.exists(nii_path_t1):
                    nii_path_t1 = os.path.join(data_dir, sub, f"{sub}_T1_MNI152_1mm.nii.gz")
            
            # CERCA IMMAGINE FLAIR
            nii_path_flair = os.path.join(data_dir, sub, f"{sub}_FLAIR_MNI152_1mm.nii")
            if not os.path.exists(nii_path_flair):
                nii_path_flair += ".gz"
            
            # Verifica esistenza json e scansioni
            json_path = os.path.join(data_dir, sub, f"{sub}_participant_info.json")
            if not os.path.exists(json_path) or not os.path.exists(nii_path_t1) or not os.path.exists(nii_path_flair):
                continue
                
            with open(json_path, 'r') as f:
                info = json.load(f)
                
            try:
                group = info['participant_info']['group'].lower()
                unique_groups.add(group)
                raw_samples.append({
                    'nii_path_t1': nii_path_t1, 
                    'nii_path_flair': nii_path_flair, 
                    'group': group,
                    'sub_id': sub
                })
            except KeyError:
                continue
                
        # Manteniamo la mappatura ORIGINALE usata nel training (Alfabetica: 0=fcd, 1=hc)
        # Invertiremo le label SOLO per il plot delle metriche, in modo da non rovinare il caricamento dei pesi!
        unique_groups = sorted(list(unique_groups))
        self.class_map = {g: i for i, g in enumerate(unique_groups)}
        print(f"Mappatura di Base (Come addestrata): {self.class_map}")
        
        for s in raw_samples:
            self.samples.append({
                'nii_path_t1': s['nii_path_t1'],
                'nii_path_flair': s['nii_path_flair'],
                'label': self.class_map[s['group']],
                'sub_id': s['sub_id']
            })
            
        print(f"Trovati {len(self.samples)} pazienti completi di T1 e FLAIR.")

    def __len__(self):
        return len(self.samples)
        
    def _process_image(self, nii_path):
        img = nib.load(nii_path)
        data = img.get_fdata(dtype=np.float32)
        
        mean_val = np.mean(data)
        if mean_val > 0:
            data = data / mean_val
            
        in_sp = data.shape
        out_sp = (160, 192, 160)
        x_c = int((in_sp[0] - out_sp[0]) / 2)
        y_c = int((in_sp[1] - out_sp[1]) / 2)
        z_c = int((in_sp[2] - out_sp[2]) / 2)
        
        data = data[x_c:x_c+out_sp[0], y_c:y_c+out_sp[1], z_c:z_c+out_sp[2]]
        data = np.expand_dims(data, axis=0)
        return torch.from_numpy(data)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        t_t1 = self._process_image(sample['nii_path_t1'])
        t_flair = self._process_image(sample['nii_path_flair'])
        label = torch.tensor(sample['label'], dtype=torch.long)
        return t_t1, t_flair, label, sample['sub_id']

## 3. Isolamento del Test Set

In [ ]:
dataset = OpenNeuroMultiModalDataset(KAGGLE_DATA_DIR)
dataset_size = len(dataset)

if dataset_size > 0:
    all_labels = [sample['label'] for sample in dataset.samples]
    all_indices = np.arange(dataset_size)
    
    # Stesso random_state usato nel training per isolare lo stesso Test Set
    _, test_idx, _, _ = train_test_split(
        all_indices, all_labels, 
        test_size=0.20, random_state=42, stratify=all_labels
    )
    
    test_dataset = torch.utils.data.Subset(dataset, test_idx)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)
    print(f"Pazienti Isolati nel Test Set: {len(test_dataset)}")

## 4. Inizializzazione Modelli

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_model(path):
    model = SFCN(output_dim=OUTPUT_DIM, channel_number=CUSTOM_CHANNELS)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model.load_state_dict(torch.load(path, map_location=device))
    model = model.to(device)
    model.eval()
    return model

if dataset_size > 0:
    print("Caricamento Modello T1...")
    model_t1 = load_model(MODEL_T1_PATH)
    
    print("Caricamento Modello FLAIR...")
    model_flair = load_model(MODEL_FLAIR_PATH)
    
    print("Modelli caricati correttamente.")

## 5. Inferenza ed Ensemble (con Mappatura Clinica Corretta)

In [ ]:
if dataset_size > 0:
    true_labels_original = []
    probs_t1_fcd = []
    probs_flair_fcd = []
    probs_ens_fcd = []
    
    with torch.no_grad():
        for t_t1, t_flair, labels, sub_ids in test_loader:
            t_t1, t_flair = t_t1.to(device), t_flair.to(device)
            true_labels_original.append(labels.item())
            
            # Predizione T1
            out_t1 = model_t1(t_t1)[0].reshape(1, -1)
            prob_t1 = torch.exp(out_t1).cpu().numpy()[0]
            probs_t1_fcd.append(prob_t1[0]) # Indice 0 è FCD nell'addestramento originale
            
            # Predizione FLAIR
            out_flair = model_flair(t_flair)[0].reshape(1, -1)
            prob_flair = torch.exp(out_flair).cpu().numpy()[0]
            probs_flair_fcd.append(prob_flair[0])
            
            # Ensemble
            mean_prob_fcd = (prob_t1[0] + prob_flair[0]) / 2.0
            probs_ens_fcd.append(mean_prob_fcd)
            
    # --- CORREZIONE DELLE LABEL PER LA STAMPA DELLE METRICHE ---
    # Vogliamo che HC sia 0 e FCD sia 1 per calcoli clinici corretti.
    # Nel training avevamo: fcd=0, hc=1.
    y_true_clinical = [1 if y == 0 else 0 for y in true_labels_original]
    
    def evaluate_clinical(probs_fcd, name):
        # Se probabilità di fcd > 0.5, diciamo 1 (fcd), altrimenti 0 (hc)
        preds = [1 if p > 0.5 else 0 for p in probs_fcd]
        acc = accuracy_score(y_true_clinical, preds)
        try:
            auc = roc_auc_score(y_true_clinical, probs_fcd)
        except:
            auc = float('nan')
        return acc, auc, preds
        
    acc_t1, auc_t1, preds_t1 = evaluate_clinical(probs_t1_fcd, "T1")
    acc_fl, auc_fl, preds_fl = evaluate_clinical(probs_flair_fcd, "FLAIR")
    acc_en, auc_en, preds_en = evaluate_clinical(probs_ens_fcd, "Ensemble")
    
    print("\n>>> RISULTATI FINALI (Clinicamente Mappati: HC=0, FCD=1) <<<")
    print(f"Modello T1     -> Acc: {acc_t1:.4f} | AUC: {auc_t1:.4f}")
    print(f"Modello FLAIR  -> Acc: {acc_fl:.4f} | AUC: {auc_fl:.4f}")
    print(f"ENSEMBLE T1+FL -> Acc: {acc_en:.4f} | AUC: {auc_en:.4f}")
    
    # --- PLOT GRAFICI ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    labels = ['Accuracy', 'ROC-AUC']
    x = np.arange(len(labels))
    width = 0.25
    
    ax1.bar(x - width, [acc_t1, auc_t1], width, label='Solo T1', color='lightcoral', edgecolor='black')
    ax1.bar(x, [acc_fl, auc_fl], width, label='Solo FLAIR', color='skyblue', edgecolor='black')
    rects3 = ax1.bar(x + width, [acc_en, auc_en], width, label='Ensemble', color='mediumseagreen', edgecolor='black')
    
    ax1.set_ylabel('Score', fontsize=12)
    ax1.set_title('Confronto Prestazioni OpenNeuro Test Set', fontsize=14, fontweight='bold', pad=15)
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, fontsize=12)
    ax1.set_ylim(0, 1.1)
    ax1.legend()
    
    for rect in rects3:
        height = rect.get_height()
        if not np.isnan(height):
            ax1.annotate(f'{height:.3f}', xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontweight='bold')
    
    # Plot 2: Confusion Matrix
    cm = confusion_matrix(y_true_clinical, preds_en)
    class_names = ['HC (Sani)', 'FCD (Malati)']
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax2)
    ax2.set_title("Confusion Matrix - Ensemble T1+FLAIR", fontweight='bold', pad=15)
    ax2.set_xlabel("Predizione")
    ax2.set_ylabel("Reale")
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'openneuro_ensemble_multimodal_results.png'), dpi=300)
    plt.show()